In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, broadcast

# 1. Policies Table (Lookup data)
policies_data = [
    ("P01", "Health"), ("P02", "Life"), ("P03", "Auto"), ("P04", "Home"),
    ("P05", "Travel"), ("P06", "Health"), ("P07", "Life"), ("P08", "Auto")
]
policies_df = spark.createDataFrame(policies_data, ["policy_id", "policy_type"])

# 2. Claims Table (25+ Rows - Skewed Data for P01)
claims_data = [
    ("C101", "P01", 50000, "COVID"), ("C102", "P01", 60000, "COVID"), 
    ("C103", "P01", 55000, "COVID"), ("C104", "P01", 45000, "COVID"),
    ("C105", "P01", 70000, "COVID"), ("C106", "P01", 65000, "COVID"),
    ("C107", "P01", 52000, "COVID"), ("C108", "P01", 58000, "COVID"),
    ("C109", "P01", 61000, "COVID"), ("C110", "P01", 49000, "COVID"),
    ("C111", "P02", 200000, "Death"), ("C112", "P02", 250000, "Death"),
    ("C113", "P03", 15000, "Accident"), ("C114", "P03", 18000, "Accident"),
    ("C115", "P04", 30000, "Fire"), ("C116", "P04", 40000, "Flood"),
    ("C117", "P05", 5000, "Delay"), ("C118", "P06", 80000, "Surgery"),
    ("C119", "P06", 90000, "Surgery"), ("C120", "P07", 300000, "Death"),
    ("C121", "P08", 12000, "Accident"), ("C122", "P01", 53000, "COVID"),
    ("C123", "P01", 62000, "COVID"), ("C124", "P01", 56000, "COVID"),
    ("C125", "P03", 17000, "Accident"), ("C126", "P06", 85000, "Surgery")
]
claims_df = spark.createDataFrame(claims_data, ["claim_id", "policy_id", "claim_amount", "reason"])

print("Data generation successful!")

Data generation successful!


In [0]:
# NARROW TRANSFORMATION: Just filtering data 
high_value_claims = claims_df.filter(col("claim_amount") > 20000)

# WIDE TRANSFORMATION: Grouping data (Causes data shuffle)
claims_summary = high_value_claims.groupBy("reason").sum("claim_amount")

claims_summary.show()

+-------+-----------------+
| reason|sum(claim_amount)|
+-------+-----------------+
|  COVID|           736000|
|  Death|           750000|
|  Flood|            40000|
|   Fire|            30000|
|Surgery|           255000|
+-------+-----------------+



In [0]:
# REPARTITION: Increase partitions to 4 for better parallel processing
optimized_claims = high_value_claims.repartition(4, "policy_id")

# CACHE: Save data in memory because we will use it multiple times next
optimized_claims.cache()
print(f"Total optimized claims: {optimized_claims.count()}") # Action to load into cache

# COALESCE: Reduce partitions to 1 before saving 
final_output_ready = optimized_claims.coalesce(1)

# UNPERSIST: Clear the cache once we are done
optimized_claims.unpersist()

Total optimized claims: 21


DataFrame[claim_id: string, policy_id: string, claim_amount: bigint, reason: string]

In [0]:

joined_df = optimized_claims.join(broadcast(policies_df), "policy_id", "inner")

joined_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonProject [policy_id#637, claim_id#636, claim_amount#638L, reason#639, policy_type#641]
         +- PhotonBroadcastHashJoin [policy_id#637], [policy_id#640], Inner, BuildRight, false, true
            :- PhotonShuffleExchangeSource
            :  +- PhotonShuffleMapStage REPARTITION_BY_NUM, [id=#837]
            :     +- PhotonShuffleExchangeSink hashpartitioning(policy_id#637, 4)
            :        +- PhotonRowToColumnar
            :           +- LocalTableScan [claim_id#636, policy_id#637, claim_amount#638L, reason#639]
            +- PhotonShuffleExchangeSource
               +- PhotonShuffleMapStage EXECUTOR_BROADCAST, [id=#843]
                  +- PhotonShuffleExchangeSink SinglePartition
                     +- PhotonRowToColumnar
                        +- LocalTableScan [policy_id#640, policy_type#641]


== Photon Explanation ==
The query

In [0]:

joined_df.createOrReplaceTempView("claims_view")

sql_query = """
    WITH ClaimData AS (
        SELECT policy_type, reason, claim_amount
        FROM claims_view
    )
    SELECT 
        policy_type,
        reason,
        claim_amount,
        ROW_NUMBER() OVER(PARTITION BY policy_type ORDER BY claim_amount DESC) as row_num,
        RANK() OVER(PARTITION BY policy_type ORDER BY claim_amount DESC) as rank_val,
        DENSE_RANK() OVER(PARTITION BY policy_type ORDER BY claim_amount DESC) as dense_rank_val
    FROM ClaimData
"""
final_report = spark.sql(sql_query)
final_report.show(30, truncate=False)

+-----------+-------+------------+-------+--------+--------------+
|policy_type|reason |claim_amount|row_num|rank_val|dense_rank_val|
+-----------+-------+------------+-------+--------+--------------+
|Health     |Surgery|90000       |1      |1       |1             |
|Health     |Surgery|85000       |2      |2       |2             |
|Health     |Surgery|80000       |3      |3       |3             |
|Health     |COVID  |70000       |4      |4       |4             |
|Health     |COVID  |65000       |5      |5       |5             |
|Health     |COVID  |62000       |6      |6       |6             |
|Health     |COVID  |61000       |7      |7       |7             |
|Health     |COVID  |60000       |8      |8       |8             |
|Health     |COVID  |58000       |9      |9       |9             |
|Health     |COVID  |56000       |10     |10      |10            |
|Health     |COVID  |55000       |11     |11      |11            |
|Health     |COVID  |53000       |12     |12      |12         

In [0]:
access_key = "HERE_AWS_ACCESS_KEY"
secret_key = "HERE_AWS_SECRET_KEY"
bucket_name = "insurance-project-prasad"

spark.conf.set("fs.s3a.access.key", access_key)
spark.conf.set("fs.s3a.secret.key", secret_key)
spark.conf.set("fs.s3a.endpoint", "s3.amazonaws.com")

print("AWS Credentials setup successful!")

s3_path = f"s3a://{bucket_name}/insurance_project/final_output/"

final_report.write \
    .format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .save(s3_path)

print(f"Data successful ga AWS S3 lo save ayyindi: {s3_path}")

AWS Credentials setup successful!
Data successful ga AWS S3 lo save ayyindi: s3a://insurance-project-prasad/insurance_project/final_output/
